# Task 2 — Define the prediction and evaluation setting 
Formally define the classification problem and the conditions under which the proposed solution will be 
evaluated. 
 
Begin by specifying: 
– the unit of observation; 
– the positive and negative classes; 
– how the target is derived from end_cause; 
– the moment at which the prediction is intended to be made; 
– the purpose for which the prediction could be used; 
– the operational meaning of predicting a session as abnormal. 
 
Determine which supplied variables would genuinely be available at the selected prediction time. Classify the 
variables into the following categories: 
1. variables eligible for use as model predictors; 
2. variables used exclusively to construct the target; 
3. variables excluded because they are generated during or after the session; 
4. variables excluded for another methodological reason. 
Every exclusion must be justified.  
 
Evaluation objectives and error consequences 
Analyse the consequences of the two main classification errors: 
– a false positive, in which a normal session is incorrectly flagged as abnormal; 
– a false negative, in which an abnormal session is not detected. 
 
Discuss which type of error is likely to be more important in the proposed application and why. Consider the 
actions that might follow an alert, such as automated monitoring, technical inspection, maintenance 
prioritisation, or user intervention. The relative importance of these errors should guide the selection of 
evaluation metrics and, later, the decision threshold.

Selection of evaluation metrics 
Define an evaluation framework appropriate for the class distribution and operational objective. Select: 
– a primary metric for model comparison and optimisation; 
– complementary predictive metrics; 
– operational metrics that describe the practical consequences of the predictions; 
– measures for evaluating performance stability across temporal periods. 
 
The selected metrics must be justified before model optimisation begins. The justification should explain what 
each metric measures, why it is relevant, and what limitations it has in this problem. 
 
Beside model evaluation metrics you may additionally consider operational evaluation metrics: 
– correct alerts per 1,000 sessions; 
– false alerts per 1,000 sessions; 
– missed abnormal sessions per 1,000 sessions. 
 
This list provides guidance and is not a substitute for reasoned metric selection. Students may propose 
additional metrics when they are relevant to the defined operational setting. 
 
Design a temporal evaluation strategy 
The observations must not be randomly divided without considering time. Training observations must precede 
the observations on which the model is evaluated. 
Design a temporal validation strategy that supports: 
– model comparison; 
– hyperparameter selection; 
– decision-threshold selection; 
– unbiased final evaluation. 
 
An expanding-window or rolling-origin strategy is recommended for model development. A final chronologically 
later test period must remain untouched until all modelling decisions have been completed. 
 
Explain and justify: 
– the selected temporal boundaries; 
– the number of validation folds; 
– any differences in fold sizes; 
– how preprocessing is fitted inside each training fold; 
– how the final test set is protected from model-selection decisions. 
 
Aggregation across temporal folds 
Explain how performance will be summarised across temporal evaluation periods. The evaluation plan should 
consider: 
– mean performance; 
– variability across folds; 
– differences in fold sizes; 
– weighted, unweighted, or pooled results; 
– possible deterioration or improvement over time. 

A model should not be judged solely by its average score if its performance is unstable across temporal periods. 
At the end of this task, present a concise evaluation protocol specifying: 
– the admissible predictor set; 
– the variables excluded and the reasons for exclusion; 
– the selected primary and complementary metrics; 
– the operational measures; 
– the baseline strategy; 
– the method used to aggregate and compare results across temporal folds. 
 
This protocol must be established before hyperparameter optimisation and final model selection. 
 
 
**Establish an appropriate baseline**
 
Define the reference against which model performance will be assessed. This should include an appropriate no-
skill or simple baseline and a correct interpretation of its expected performance under the selected metrics.

In [223]:
import pandas as pd


In [224]:
file_path = r"C:\Users\Raphael\Desktop\Mineração de Dados\Minera-o_de_Dados_MEI\data\Charging_Data_tratado.csv"
df = pd.read_csv(file_path)

df.shape

(441077, 32)

In [225]:
df['Start Time'] = pd.to_datetime(df['Start Time'])

df = df.sort_values('Start Time').reset_index(drop=True)

df['Start Time'].dt.month 

0         12
1         12
2         12
3         12
4         12
          ..
441072    12
441073    12
441074    12
441075    12
441076    12
Name: Start Time, Length: 441077, dtype: int32

In [226]:
df['is_Abnormal'].value_counts()

is_Abnormal
0    359143
1     81934
Name: count, dtype: int64

In [227]:
# Aqui separamos uma parcela do dataframe 
# para poder usar da forma que queremos
teste = df[df['Start Time'] >= '2021-10-01']
teste.shape


(80081, 32)

In [228]:
df[(df['Start Time'] >= '2021-02-01') & (df['Start Time'] < '2021-04-01')]['is_Abnormal'].sum()


np.int64(3763)

In [229]:
df[(df['Start Time'] >= '2020-02-01') & (df['Start Time'] < '2020-10-01')].shape


(119290, 32)

In [230]:
treino_1 = df[(df['Start Time'] >= '2020-02-01') & (df['Start Time'] < '2021-01-01')]
treino_1.shape

(179654, 32)

In [231]:
validacao_1 = df[(df['Start Time'] >= '2021-01-01') & (df['Start Time'] < '2021-04-01')]
validacao_1.shape

(41797, 32)

In [232]:
validacao_1['is_Abnormal'].sum()

np.int64(6611)

In [233]:
treino_2 = df[(df['Start Time'] >= '2020-05-01') & (df['Start Time'] < '2021-04-01')]
treino_2.shape

(202532, 32)

In [234]:
validacao_2 = df[(df['Start Time'] >= '2021-04-01') & (df['Start Time'] < '2021-07-01')]
validacao_2.shape

(57542, 32)

In [235]:
validacao_2['is_Abnormal'].sum()

np.int64(10487)

In [236]:
treino_3 = df[(df['Start Time'] >= '2020-08-01') & (df['Start Time'] < '2021-07-01')]
treino_3.shape

(205496, 32)

In [237]:
validacao_3 = df[(df['Start Time'] >= '2021-07-01') & (df['Start Time'] < '2021-10-01')]
validacao_3.shape


(75769, 32)

In [238]:
validacao_3['is_Abnormal'].sum()

np.int64(16016)

In [239]:
df.columns


Index(['Location Information', 'District Name', 'Start Time', 'Temperature(℃)',
       'Relative Humidity(%)', 'Precipitation(mm)', 'tariff_period',
       'post_previous_sessions_1D', 'post_previous_abnormal_1D',
       'post_previous_abnormal_rate_1D', 'post_previous_sessions_7D',
       'post_previous_abnormal_7D', 'post_previous_abnormal_rate_7D',
       'post_previous_abnormal_30D', 'post_previous_abnormal_rate_30D',
       'post_hours_since_previous_completed_session',
       'post_no_previous_completed_session',
       'post_hours_since_previous_completed_abnormal',
       'post_no_previous_completed_abnormal', 'user_previous_sessions_30D',
       'user_previous_abnormal_rate_30D', 'user_previous_sessions_180D',
       'user_previous_abnormal_rate_180D', 'user_previous_sessions_730D',
       'user_days_since_previous_completed_session',
       'user_no_previous_completed_session',
       'user_days_since_previous_completed_abnormal',
       'user_no_previous_completed_abnormal',

In [240]:
from sklearn.metrics import accuracy_score, recall_score

In [241]:
# "Modelo" que prevê sempre normal (0)
previsao_1 = [0] * len(validacao_1)

In [242]:
accuracy_score(validacao_1['is_Abnormal'], previsao_1)

0.8418307534033543

In [243]:
recall_score(validacao_1['is_Abnormal'], previsao_1)

0.0

In [244]:
previsao_2 = [0] * len(validacao_2)

In [245]:
accuracy_score(validacao_2['is_Abnormal'], previsao_2)

0.817750512669007

In [246]:
recall_score(validacao_2['is_Abnormal'], previsao_2)

0.0

In [247]:
previsao_3 = [0] * len(validacao_3)

In [248]:
accuracy_score(validacao_3['is_Abnormal'], previsao_3)

0.7886206760020589

In [249]:
recall_score(validacao_3['is_Abnormal'], previsao_3)

0.0

In [250]:
from sklearn.metrics import average_precision_score

In [251]:
# AUC-PR do baseline no-skill (sempre normal)
average_precision_score(validacao_1['is_Abnormal'], previsao_1)

0.1581692465966457

In [252]:
# AUC-PR do baseline de regra (taxa de falhas do posto no dia anterior)
regra_1 = average_precision_score(validacao_1['is_Abnormal'], validacao_1['post_previous_abnormal_rate_1D'])
regra_1

0.44313010672069764

In [253]:
average_precision_score(validacao_2['is_Abnormal'], previsao_2)

0.18224948733099303

In [254]:
regra_2 = average_precision_score(validacao_2['is_Abnormal'], validacao_2['post_previous_abnormal_rate_1D'])
regra_2

0.5634521379436742

In [255]:
average_precision_score(validacao_3['is_Abnormal'], previsao_3)

0.2113793239979411

In [256]:
regra_3 = average_precision_score(validacao_3['is_Abnormal'], validacao_3['post_previous_abnormal_rate_1D'])
regra_3

0.5917857266162994

In [257]:
# Agregação entre folds: média simples
(regra_1 + regra_2 + regra_3) / 3

0.5327893237602237

In [258]:
# Desvio-padrão entre os folds
pd.Series([regra_1, regra_2, regra_3]).std()

np.float64(0.07892895198767695)

In [259]:
# Média ponderada pelo tamanho de cada validação
(regra_1 * len(validacao_1) + regra_2 * len(validacao_2) + regra_3 * len(validacao_3)) / (len(validacao_1) + len(validacao_2) + len(validacao_3))

0.546992054687109

In [260]:
# Pooled: juntar as 3 validações numa só
validacoes = pd.concat([validacao_1, validacao_2, validacao_3])

In [261]:
average_precision_score(validacoes['is_Abnormal'], validacoes['post_previous_abnormal_rate_1D'])

0.5541485943277602

---

## 6. Agregação entre folds

Para ver na prática a diferença entre as formas de agregar, aplicamos todas ao baseline de regra (secção 7):

| Forma de agregar | AUC-PR da regra |
|---|---|
| Média simples | 0,533 |
| Desvio-padrão | 0,079 |
| Média ponderada pelo tamanho da validação | 0,547 |
| Pooled (as 3 validações juntas) | 0,554 |

A ponderada e a pooled acabam por puxar o resultado para cima porque o fold 3 é o maior e também o que tem o valor mais alto, ou seja, dão mais peso ao período mais recente. Logo, os modelos vão ser comparados pela **média simples**, onde cada período conta o mesmo, sempre junto com o desvio-padrão, o pior fold, o valor de cada fold e o número de folds em que o modelo passa da regra. Um desvio de 0,079 (cerca de 15% da média) mostra que a média sozinha esconde muita variação. A forma pooled fica para a escolha do threshold e para as métricas operacionais.

Quanto à evolução no tempo, a AUC-PR da regra sobe de 0,443 para 0,592, mas a taxa de anomalia também sobe (de 15,8% para 21,1%). Dividindo uma pela outra a regra fica praticamente estável (2,8, 3,1 e 2,8 vezes o no-skill), ou seja, a melhora vem do período e não da regra, e por isso a evolução tem que ser lida sempre em relação ao baseline de cada fold.

---

## 7. Baseline

| Fold | Taxa de anomalia | No-skill (AUC-PR) | Regra (AUC-PR) |
|---|---|---|---|
| 1 | 15,8% | 0,158 | 0,443 |
| 2 | 18,2% | 0,182 | 0,564 |
| 3 | 21,1% | 0,211 | 0,592 |

- **No-skill:** prevê sempre normal. Como não ordena as sessões, a sua AUC-PR é exatamente a taxa de anomalia de cada fold, o que confirmamos nos resultados. Qualquer modelo abaixo disso não aprendeu nada.
- **Regra simples:** usa diretamente a taxa de falhas do posto no dia anterior (post_previous_abnormal_rate_1D) como pontuação de risco. Sem treino nem threshold, já é praticamente 3 vezes melhor que o no-skill.

Logo, a régua que realmente importa é a regra. Se o modelo não passar claramente dela, as outras variáveis (clima, localização, histórico do usuário, etc.) não estão a acrescentar nada ao que o histórico do posto já diz sozinho.

---

## 8. Protocolo de avaliação

Este protocolo fica definido antes de qualquer modelo ser treinado e não vai ser alterado depois de ver os resultados.

| Item | Decisão |
|---|---|
| Preditores admitidos | as 28 colunas do CSV tratado e o Start Time (hora e dia da semana) |
| Excluídos | End cause (só para o alvo), as 7 colunas pós-sessão, Order creation time, electricity_price e as 6 variáveis redundantes |
| A avaliar na modelação | UserID e Charging Post ID (com e sem) |
| Métrica principal | AUC-PR |
| Complementares | recall, precision, F1, balanced accuracy, MCC e ROC-AUC |
| Operacionais | alertas corretos, alertas falsos e anomalias perdidas a cada 1 000 sessões |
| Threshold | maior precision com recall mínimo, escolhido nas 3 validações juntas |
| Baselines | no-skill (AUC-PR = taxa de anomalia do fold) e regra com a post_previous_abnormal_rate_1D |
| Validação | rolling-origin, 11 meses de treino, 3 folds trimestrais em 2021; teste em out–dez/2021 usado uma única vez |
| Agregação | média simples da AUC-PR com desvio-padrão, pior fold e folds ganhos da regra; pooled para threshold e métricas operacionais |
| Comparação estatística | nos folds pela média, desvio e folds ganhos; no teste final com o McNemar |

Lembrando que um modelo só vai ser considerado bom se passar claramente da regra em todos os folds, e não só na média.
